# Layered unstructured grids (DISU) with vorflow

`vorflow.disu` turns vorflow plan grids and FloPy-style layer arrays (`top`, `botm`, `idomain`) into a MODFLOW DISU grid: one node per active cell, and every connection between cells with its lengths and areas. It offers three tools, which can be combined:

| Tool | Cell walls | Suits |
|---|---|---|
| **The same plan grid in every layer** (as in DISV) | fixed in plan through the whole column; layers follow any surface | subhorizontal features: strata, aquitards, a pit floor |
| **A different plan grid per layer** | can move from one layer to the next | subvertical features, locally: pit walls, faults, bores |
| **Sublayers in some columns** | as either of the above | local vertical detail |

Most of a model usually uses the first; the second refines the area around a subvertical feature, only in the layers it cuts; the third adds vertical detail where it is needed. This notebook builds each one on the same site and shows what vorflow does with the cells. Nothing here runs MODFLOW: the last section shows the arrays that go to FloPy.

In [ ]:
import tempfile
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import LineString, Point, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, disu
from vorflow.utils import build_connectivity, calculate_mesh_quality

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

workspace = Path(tempfile.mkdtemp(prefix="vorflow_disu_"))

Two small helpers draw vertical sections: the active cells cut by a plan line, at their model elevations (after pass-through intervals are closed), and optionally the connections between their centres.

In [ ]:
def section_cells(d, line):
    # Active cells cut by a plan line: node, layer, start and end along the line, bottom and top.
    props = d.get_gridprops_mf6_disu()
    grids = d.grids_per_layer
    rows = []
    for row in d.cellmap[d.cellmap.node >= 0].itertuples():
        cut = grids[row.layer].geometry.iloc[row.icpl].intersection(line)
        if cut.is_empty or cut.length == 0:
            continue
        s = [line.project(Point(xy)) for xy in shapely.get_coordinates(cut)]
        rows.append((row.node, row.layer, min(s), max(s), props["bot"][row.node], props["top"][row.node]))
    return pd.DataFrame(rows, columns=["node", "layer", "s0", "s1", "bot", "top"])


def draw_cells(ax, cells, colors, hatches=None, lw=0.4, edgecolor="k"):
    for i, c in enumerate(cells.itertuples()):
        ax.add_patch(Rectangle((c.s0, c.bot), c.s1 - c.s0, c.top - c.bot, facecolor=colors[i], edgecolor=edgecolor,
                               lw=lw, hatch=None if hatches is None else hatches[i]))
    ax.autoscale_view()


def draw_links(ax, d, cells):
    # Connections between the centres of the drawn cells, by kind; returns legend handles.
    centre = {c.node: ((c.s0 + c.s1) / 2, (c.bot + c.top) / 2) for c in cells.itertuples()}
    layer = dict(zip(cells.node, cells.layer))
    kinds = {"vertical": [], "same layer": [], "between layers": []}
    for n, m, ihc in zip(d.links["n"], d.links["m"], d.links["ihc"]):
        if n in centre and m in centre:
            kind = "vertical" if ihc == 0 else "same layer" if layer[n] == layer[m] else "between layers"
            kinds[kind].append([centre[n], centre[m]])
    style = {"vertical": ((0.1, 0.55, 0.2), 1.6, (0, (3, 2))), "same layer": ((0.0, 0.0, 0.0), 1.4, "solid"),
             "between layers": ((1.0, 0.5, 0.05), 2.2, "solid")}
    handles = []
    for kind, segments in kinds.items():
        if segments:
            color, width, dash = style[kind]
            ax.add_collection(LineCollection(segments, colors=[color], linewidths=width, linestyles=[dash]))
            handles.append(Line2D([], [], color=color, lw=width, ls=dash, label=f"connection, {kind}"))
    xy = np.array(list(centre.values()))
    ax.plot(xy[:, 0], xy[:, 1], "o", color="k", ms=2.5)
    handles.append(Line2D([], [], color="k", marker="o", ms=4, ls="none", label="cell centre"))
    return handles

## 1. The same plan grid in every layer

This is the grid of a DISV model: one plan grid repeated in every layer, with layers that follow the strata. Building it as DISU adds two things:

- **Connections between offset cells.** DISV connects a cell to its neighbours in the same layer and to the cells directly above and below. `DisuGrid` also connects side-by-side cells of different layers wherever their walls overlap vertically. This matters as soon as layers dip (next subsection).
- **A base for the other tools.** Pinch-outs (section 2), a different grid per layer (section 3) and sublayers (section 4) all start from it.

The site is 1000 m × 600 m with 60 m Voronoi cells. A pit (200 m × 200 m) is a barrier line, so its outline is made of cell faces; it is used again in sections 3 and 4. Five layers dip 3 % to the east. `DisuGrid` takes the plan grid as `VoronoiTessellator.generate()` returns it (a GeoDataFrame with one polygon per cell and its centre in `x`/`y`), and `top` and `botm` as FloPy does.

In [ ]:
domain = box(0, 0, 1000, 600)
pit = box(400, 200, 600, 400)


def mesh(fine_pit=None):
    cm = ConceptualMesh()
    cm.add_polygon(domain, zone_id=1, resolution=60)
    cm.add_line(LineString(pit.exterior.coords), line_id="pit", resolution=30, is_barrier=True)
    if fine_pit:
        cm.add_polygon(pit, zone_id=2, resolution=fine_pit, embed=False)
    mg = MeshGenerator(background_lc=60, conceptual_mesh=cm, verbosity=0)
    mg.generate()
    vt = VoronoiTessellator(mg, cm)
    return vt, vt.generate()


vt, base = mesh()
xb = base.x.to_numpy()
top = 100.0 - 0.03 * xb
depths = (20.0, 40.0, 50.0, 60.0, 80.0)
botm = [100.0 - depth - 0.03 * xb for depth in depths]
d1 = disu.DisuGrid(base, top, botm).build()

section_line = LineString([(0, 300), (1000, 300)])
layer_colors = ["#eff3ff", "#c6dbef", "#9ecae1", "#6baed6", "#2171b5"]
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1, 1.3]})
base.boundary.plot(ax=axes[0], color="0.55", linewidth=0.5)
axes[0].plot(*pit.exterior.xy, color="tab:red", lw=1.8)
axes[0].plot(*section_line.xy, color="k", ls="--", lw=1.2)
axes[0].set(xlim=(0, 1000), ylim=(0, 600), xlabel="x (m)", ylabel="y (m)",
            title=f"Plan: {len(base)} cells, the same in every layer")
axes[0].set_aspect("equal")
axes[0].legend(handles=[Line2D([], [], color="0.55", lw=0.6, label="cells"),
                        Line2D([], [], color="tab:red", lw=1.8, label="pit outline (cell faces)"),
                        Line2D([], [], color="k", ls="--", lw=1.2, label="section on the right")],
               loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False)
cells = section_cells(d1, section_line)
axes[1].axvspan(400, 600, color="tab:red", alpha=0.08, lw=0)
draw_cells(axes[1], cells, [layer_colors[k] for k in cells.layer], edgecolor="0.35")
axes[1].set_aspect(5)
axes[1].set(xlabel="x (m)", ylabel="z (m)", title="Section along y = 300 m (vertical exaggeration 5×)")
axes[1].legend(handles=[Patch(facecolor=c, edgecolor="0.35", label=f"layer {k + 1}") for k, c in enumerate(layer_colors)]
               + [Patch(facecolor="tab:red", alpha=0.15, label="below the pit")],
               loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=6, frameon=False)
plt.show()
print(d1.report())

Every column has the same five layers, and each layer follows the dip, so side-by-side cells are offset by 3 % of the distance between their centres. `report()` lists the connections: vertical ones within each column, and side-by-side ones, all written as staggered (IHC=2) because with full connectivity any of them may join offset cells.

**From an existing DISV model.** A plan grid that does not come from vorflow, such as an existing FloPy or LoopFlopy DISV model or a grid from another mesher, goes through `DisuGrid.from_disv` instead. It takes the `vertices` and `cell2d` arrays of `flopy.mf6.ModflowGwfdisv` and builds the same DISU grid. Here the arrays come from the tessellator, so the result can be checked against `d1`.

In [ ]:
disv = vt.to_disv()
d1_disv = disu.DisuGrid.from_disv(disv["vertices"], disv["cell2d"], top, botm).build()
print("nodes:", d1.nodes, d1_disv.nodes, "| connections:", len(d1.links["n"]), len(d1_disv.links["n"]))

### Layered or full connectivity

Layers that follow dipping strata are offset from their neighbours. DISV (and `connectivity="layered"`) connects a cell only to the neighbours in its own layer, so water following a dipping aquifer has to step through the units above and below it. `connectivity="full"` (the default) connects every pair of neighbouring cells whose side faces overlap vertically, as Langevin's USGS `disv2disu.py` does (Provost et al. 2025, https://doi.org/10.1111/gwat.13459).

The section below is a 30° dipping aquifer (blue, 3 layers) between clay layers, on a strip of 1 m columns.

In [ ]:
def strip(n, width=1.0):
    polygons = [box(i * width, 0, (i + 1) * width, 1) for i in range(n)]
    return gpd.GeoDataFrame({"x": [p.centroid.x for p in polygons], "y": 0.5}, geometry=polygons)


def dipping_strip(width=1.0, length=11.0, dip=30.0):
    grid = strip(int(round(length / width)), width)
    rise = np.tan(np.radians(dip)) * grid.x.to_numpy()
    surfaces = rise[None, :] + np.arange(6, -4, -1)[:, None]      # nine 1 m layers; 3 to 5 are the aquifer
    return grid, surfaces


grid, surfaces = dipping_strip()
line = LineString([(0, 0.5), (11, 0.5)])
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
for ax, connectivity in zip(axes, ("layered", "full")):
    d = disu.DisuGrid(grid, surfaces[0], surfaces[1:], connectivity=connectivity).build()
    cells = section_cells(d, line)
    draw_cells(ax, cells, ["#9ecae1" if 3 <= k <= 5 else "#d8c8a8" for k in cells.layer], edgecolor="0.6")
    handles = draw_links(ax, d, cells)
    ax.set_title(f'connectivity="{connectivity}": {len(d.links["n"])} connections')
    ax.set_xlabel("x (m)")
    ax.set_aspect("equal")
axes[0].set_ylabel("z (m)")
fig.legend(handles=[Patch(facecolor="#9ecae1", edgecolor="k", label="aquifer layers"),
                    Patch(facecolor="#d8c8a8", edgecolor="k", label="clay layers")] + handles,
           loc="lower center", ncol=3, frameon=False)
fig.subplots_adjust(bottom=0.22)
plt.show()
print(d.report())

With layered connectivity, every horizontal connection joins two cells of the same layer, so the aquifer's cells meet clay beside them; no formulation recovers the dip from that (Bardot et al. 2022). Full connectivity adds the orange connections between offset cells, and water can follow the aquifer. In MODFLOW 6, XT3D then reproduces flow along the dip, provided NPF is given `angle2` and `angle3` (zeros are fine): without them XT3D ignores vertical offsets. MODFLOW-USG has no XT3D; ghost nodes take its place (see `disu_faults_and_gnc.ipynb`).

### What a staircase leaves open

Each cell has a flat top, so a dipping aquifer is a staircase. Across a column boundary, aquifer meets aquifer only where the two columns' aquifer cells overlap vertically: over b − Δx·tanθ of the aquifer's vertical thickness b (Δx column width, θ dip). The rest of the face meets clay. The connections record exactly this overlap (`links["height"]`), so the lost fraction can be read from the grid:

In [ ]:
def open_height(width):
    grid, surfaces = dipping_strip(width)
    d = disu.DisuGrid(grid, surfaces[0], surfaces[1:]).build()
    cm = d.cellmap[d.cellmap.node >= 0].set_index("node")
    x, layer = cm.x.reindex(range(d.nodes)).to_numpy(), cm.layer.reindex(range(d.nodes)).to_numpy()
    aquifer = (layer >= 3) & (layer <= 5)
    n, m, height = d.links["n"], d.links["m"], d.links["height"]
    across = (d.links["ihc"] != 0) & (np.minimum(x[n], x[m]) < 5) & (np.maximum(x[n], x[m]) > 5)
    return height[across & aquifer[n] & aquifer[m]].sum(), d


widths = [1.0, 0.5, 0.25]
b, tan = 3.0, np.tan(np.radians(30))
rows = []
for w in widths:
    opened, _ = open_height(w)
    rows.append({"column width (m)": w, "open height (m)": round(opened, 3), "lost (%)": round(100 * (1 - opened / b), 1),
                 "Δx·tanθ/b (%)": round(100 * w * tan / b, 1)})
print(pd.DataFrame(rows).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 1.3]})
grid, surfaces = dipping_strip(1.0)
d = disu.DisuGrid(grid, surfaces[0], surfaces[1:]).build()
cells = section_cells(d, LineString([(0, 0.5), (11, 0.5)]))
draw_cells(axes[0], cells, ["#9ecae1" if 3 <= k <= 5 else "#d8c8a8" for k in cells.layer])
left = cells[np.isclose(cells.s1, 5) & cells.layer.between(3, 5)]
right = cells[np.isclose(cells.s0, 5) & cells.layer.between(3, 5)]
lo, hi = max(left.bot.min(), right.bot.min()), min(left.top.max(), right.top.max())
axes[0].plot([5, 5], [lo, hi], color="tab:blue", lw=6)
axes[0].plot([5, 5], [hi, max(left.top.max(), right.top.max())], color="tab:red", lw=6)
axes[0].plot([5, 5], [min(left.bot.min(), right.bot.min()), lo], color="tab:red", lw=6)
axes[0].set(xlim=(3, 7), ylim=(0, 10), title="Across the column boundary at x = 5 m", xlabel="x (m)", ylabel="z (m)", aspect="equal")
axes[0].legend(handles=[Line2D([], [], color="tab:blue", lw=6, label=f"aquifer meets aquifer: {hi - lo:.2f} m of b = 3 m"),
                        Line2D([], [], color="tab:red", lw=6, label="aquifer meets clay"),
                        Patch(facecolor="#9ecae1", edgecolor="k", label="aquifer layers"),
                        Patch(facecolor="#d8c8a8", edgecolor="k", label="clay layers")],
               loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=9, frameon=False)
ws = np.linspace(0.1, 1.0, 50)
axes[1].plot(ws, 100 * ws * tan / b, color="0.4", label="Δx·tanθ/b")
axes[1].plot(widths, [r["lost (%)"] for r in rows], "o", color="tab:red", label="read from the connections")
axes[1].axhline(5, color="tab:green", ls="--", label="5 % guidance")
axes[1].set(xlabel="column width Δx (m)", ylabel="aquifer face lost at each boundary (%)", title="Lost face against column width")
axes[1].legend()
plt.tight_layout()
plt.show()

The lost face is not a flow error in the cells: heads, flow direction and flux per square metre can all be right. It is water: the staircase carries about Δx·tanθ/b less than the smooth aquifer would, and only narrower columns reduce it (more layers do not). **Where the water carried by a dipping unit matters, keep Δx·tanθ/b at or below about 5 %**, and judge the model by its total flows against a finer grid rather than by its heads. `report()` also gives the *offset ratio* (offset / cell thickness): at 1 or more, neighbours in one layer no longer touch.

## 2. Pinch-outs and pass-through

Units that thin out, or lenses, must leave a column without breaking it. Cells with idomain -1, or thinner than `min_thickness`, are removed and the cells above and below connect directly. MODFLOW 6's own pass-through leaves their interval as a gap (`collapse=None` here); by default vorflow gives it to the cells above and below (`collapse="split"`), so the column has no gap and vertical connection lengths stay exact.

Here layer 2 has a lens (idomain -1) in the west, and layer 4 thins to nothing towards the east (cells under 1 m are removed with `min_thickness=1`).

In [ ]:
lens = np.hypot(xb - 250, base.y.to_numpy() - 300) < 90
botm_pinch = [b.copy() for b in botm]
botm_pinch[3] = botm[2] - np.clip(10 * (1 - xb / 700), 0, 10)     # layer 4: 10 m in the west, nothing east of x = 700 m
idomain = [np.ones(len(base), int) for _ in botm]
idomain[1] = np.where(lens, -1, 1)
line = LineString([(0, 300), (1000, 300)])
layer_colors = plt.cm.tab10(np.arange(5))

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, collapse in zip(axes, (None, "split")):
    d2 = disu.DisuGrid(base, top, botm_pinch, idomain, min_thickness=1.0, collapse=collapse).build()
    cells = section_cells(d2, line)
    draw_cells(ax, cells, [layer_colors[k] for k in cells.layer])
    ax.set_title(f"collapse={collapse!r}: {d2.nodes} nodes")
    ax.set_ylabel("z (m)")
axes[1].set_xlabel("x along y = 300 m (m)")
fig.legend(handles=[Patch(facecolor=layer_colors[k], edgecolor="k", label=f"layer {k + 1}") for k in range(5)],
           loc="center right", frameon=False)
fig.subplots_adjust(right=0.88)
plt.show()
print(d2.report())

The report flags the thin cells next to the pinch-out: there, the 3 % dip offsets neighbours by more than a cell's thickness, so they touch only through other layers. Full connectivity still joins them; sublayers or smaller cells would keep each layer continuous.

`idomain_layers` gives each layer's cells as MODFLOW 6 codes (1 active, -1 pass-through, 0 no-flow), on that layer's plan grid, for maps:

In [ ]:
status = d2.idomain_layers
codes = {1: ("active", "#74c476"), -1: ("pass-through (removed, interval closed)", "#fd8d3c"), 0: ("no-flow", "0.6")}
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, k in zip(axes, (1, 3)):
    base.plot(ax=ax, color=[codes[int(v)][1] for v in status[k]], edgecolor="k", linewidth=0.2)
    ax.set(title=f"layer {k + 1}", xlabel="x (m)")
    ax.set_aspect("equal")
axes[0].set_ylabel("y (m)")
fig.legend(handles=[Patch(facecolor=c, edgecolor="k", label=name) for name, c in codes.values()],
           loc="lower center", ncol=3, frameon=False)
fig.subplots_adjust(bottom=0.2)
plt.show()

## 3. A different plan grid per layer

Below 40 m the pit is meshed with 12 m cells, only in the layers it cuts. Two functions make the local grid; both keep every base cell outside the pit:

- `disu.splice` joins a separately meshed fine grid inside the outline. The nodes along the outline need not match, so some faces there are far from orthogonal.
- `disu.refill` re-meshes the inside from the base grid and shares every outline vertex, so the outline faces stay orthogonal. Here the 12 m inside cells cannot grow to the outline's 28.6 m spacing within the pit, and `refill` warns that they jump in size there.
- `disu.refill(..., outline_split=2)` halves each outline face on the inside, so the inside cells can grow smoothly. The extra vertices hang on the outside cells' faces, as in a quadtree grid: those faces are about 20° off, which XT3D handles exactly.

Each cell is coloured by its worst face's orthogonality error (the angle between the line joining two cell centres and the normal of their shared face).

In [ ]:
import warnings

_, fine = mesh(fine_pit=12)
spliced = disu.splice(base, fine, pit)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    refilled_1 = disu.refill(base, pit, resolution=12, growth_factor=1.2)
print("refill warns:", *[str(w.message).split(".")[0] for w in caught])
refilled = disu.refill(base, pit, resolution=12, growth_factor=1.2, outline_split=2)


def orthogonality(grid):
    return calculate_mesh_quality(grid, calc_ortho=True, connectivity=build_connectivity(grid, center="generator"))


fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), sharey=True)
panels = (("splice", spliced), ("refill", refilled_1), ("refill, outline_split=2", refilled))
for ax, (name, g) in zip(axes, panels):
    q = orthogonality(g)
    q.plot(ax=ax, column="ortho_error", cmap="Reds", vmin=0, vmax=40, edgecolor="k", linewidth=0.3)
    ax.plot(*pit.exterior.xy, color="tab:blue", lw=1.5)
    ax.set(xlim=(360, 520), ylim=(160, 300), xlabel="x (m)", title=f"{name}\n{len(g)} cells, worst face {q.ortho_error.max():.1f}° off")
    ax.set_aspect("equal")
axes[0].set_ylabel("y (m)")
for ax in axes[1:]:
    ax.set_ylabel("")
cbar = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0, 40), cmap="Reds"), ax=axes, shrink=0.8)
cbar.set_label("orthogonality error of the cell's worst face (°)")
axes[0].legend(handles=[Line2D([], [], color="tab:blue", lw=1.5, label="pit outline")], loc="upper left")
plt.show()

Stack the grids: the base grid in the two upper layers, the refilled grid below. Each layer's `botm` is given on that layer's grid. Cells of different grids connect through their plan overlaps.

In [ ]:
xp = refilled.x.to_numpy()
grids = [base, base, refilled, refilled, refilled]
botm3 = [100.0 - depth - 0.03 * (xb if g is base else xp) for g, depth in zip(grids, depths)]
d3 = disu.DisuGrid(grids, top, botm3).build()
print(d3.report())

# One coarse cell of layer 2 and the fine cells of layer 3 below it.
coarse = int(np.flatnonzero(base.contains(Point(500, 300)))[0])
cm3 = d3.cellmap.set_index(["layer", "icpl"])
upper = cm3.loc[(1, coarse), "node"]
lookup = d3.cellmap[(d3.cellmap.layer == 2) & (d3.cellmap.node >= 0)].set_index("node").icpl
below = [(m if n == upper else n) for n, m, ihc in zip(d3.links["n"], d3.links["m"], d3.links["ihc"])
         if ihc == 0 and upper in (n, m) and (m if n == upper else n) in lookup.index]
fig, ax = plt.subplots(figsize=(7, 6))
patches = [base.geometry.iloc[coarse].intersection(refilled.geometry.iloc[lookup[m]]) for m in below]
areas = np.array([p.area for p in patches])
gpd.GeoDataFrame({"area": areas}, geometry=patches).plot(ax=ax, column="area", cmap="viridis", edgecolor="white", linewidth=0.8)
refilled.boundary.plot(ax=ax, color="0.6", linewidth=0.5)
gpd.GeoSeries([base.geometry.iloc[coarse]]).boundary.plot(ax=ax, color="k", linewidth=2.5)
x0, y0, x1, y1 = base.geometry.iloc[coarse].bounds
ax.set(xlim=(x0 - 15, x1 + 15), ylim=(y0 - 15, y1 + 15), xlabel="x (m)", ylabel="y (m)",
       title=f"{len(below)} vertical connections from one coarse cell")
ax.set_aspect("equal")
cbar = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(areas.min(), areas.max()), cmap="viridis"), ax=ax, shrink=0.8)
cbar.set_label("connection area: plan overlap (m²)")
ax.legend(handles=[Line2D([], [], color="k", lw=2.5, label="coarse cell, layer 2"),
                   Line2D([], [], color="0.6", lw=0.5, label="fine cells, layer 3")], loc="upper left", fontsize=8)
plt.show()
print(f"overlap areas add up to {areas.sum():.1f} m², the coarse cell's area is {base.geometry.iloc[coarse].area:.1f} m²")

Where the grid changes, each cell still has one flat top. A fine cell can sit under two coarse cells, which must then share its top, and this chains across the refined area into one flat contact. vorflow makes that contact gap-free (cells overlapping across it share one elevation), so a dipping contact under the refinement becomes one level; `report()` gives how far bottoms moved (`max_contact_shift`). Section 4 shows how to keep the dip instead.

## 4. The vertical split: sublayers

`disu.subdivide` splits chosen layers, only in chosen columns: here layer 4 is split in two inside the pit. Elsewhere the extra piece has zero thickness and drops out, so the rest of the model keeps its layers. `parent` maps each new layer to the original one.

`disu.split_contacts` keeps a dipping contact at a change of grid: it cuts the contact flat and fills the space between the cut and the given contact with a wedge layer. `wedge="upper"` puts the wedge on the grid above (the coarse one here; fewest cells), `wedge="lower"` on the grid below (it follows the contact on the fine grid).

In [ ]:
inside = (refilled.splice_source == "fine").to_numpy()
lower_botm, lower_idomain, lower_parent = disu.subdivide(100.0 - 40.0 - 0.03 * xp, np.array(botm3[2:]), nsub=[1, 2, 1],
                                                         where=[None, inside, None])
grids4 = [base, base] + [refilled] * len(lower_botm)
botm4 = botm3[:2] + list(lower_botm)
idomain4 = [np.ones(len(base), int)] * 2 + [row.copy() for row in lower_idomain]
parent4 = np.r_[0, 1, 2 + lower_parent]

cases = [("flat contact (default)", (grids4, top, botm4, idomain4, np.arange(len(grids4))))]
for wedge in ("upper", "lower"):
    cases.append((f'wedge="{wedge}"', disu.split_contacts(grids4, top, botm4, idomain4, wedge=wedge)))

line = LineString([(0, 300), (1000, 300)])
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
for ax, (name, (g, t, b, i, split_parent)) in zip(axes, cases):
    d4 = disu.DisuGrid(g, t, b, i).build()
    original = parent4[split_parent]
    wedge = np.array([g[k] is not grids4[split_parent[k]] for k in range(len(g))])
    cells = section_cells(d4, line)
    draw_cells(ax, cells, [layer_colors[original[k]] for k in cells.layer],
               hatches=["///" if wedge[k] else None for k in cells.layer])
    ax.set_title(f"{name}: {d4.nodes} nodes, bottoms moved up to {d4.diagnostics['max_contact_shift']:.2f} m")
    ax.set(xlim=(300, 700), ylim=(-5, 95), ylabel="z (m)")
axes[-1].set_xlabel("x along y = 300 m (m); the pit is between 400 and 600 m")
fig.legend(handles=[Patch(facecolor=layer_colors[k], edgecolor="k", label=f"original layer {k + 1}") for k in range(5)]
           + [Patch(facecolor="white", edgecolor="k", hatch="///", label="wedge layer")], loc="center right", frameon=False)
fig.subplots_adjust(right=0.85)
plt.show()

## 5. Using the grid

`cellmap` lists every layer cell with its node (-1 when removed), layer, grid, `node_id`, status and elevations; `save_cellmap()` writes it to CSV for post-processing without vorflow. `locate` finds the node containing a point, and `column` lists the nodes under a plan point.

In [ ]:
d5 = disu.DisuGrid(grids4, top, botm4, idomain4).build()
print(d5.cellmap.head())
d5.save_cellmap(workspace / "cellmap.csv")
well = d5.locate(500.0, 300.0, 100.0 - 72.0 - 0.03 * 500.0)
print("node at (500, 300, 13):", well)
print("nodes under (500, 300):", d5.column(500.0, 300.0))

Node values map back to the layers for plotting. `layer_view` gives one value per plan cell of an *original* layer, even after it was split; here, the elevation of the middle of original layer 4, which is two sublayers inside the pit (`pick="mean"` averages them). `vertex_grid(k)` gives a FloPy grid of one layer's plan grid for `flopy.plot.PlotMapView`; no model is needed.

In [ ]:
import flopy

props = d5.get_gridprops_mf6_disu()
middle = (props["top"] + props["bot"]) / 2
k = int(np.flatnonzero(parent4 == 3)[0])       # first model layer of original layer 4
view = d5.layer_view(middle, layer=3, parent=parent4, pick="mean")
fig, ax = plt.subplots(figsize=(9, 5))
pmv = flopy.plot.PlotMapView(modelgrid=d5.vertex_grid(k), ax=ax)
pc = pmv.plot_array(view, cmap="viridis")
pmv.plot_grid(linewidth=0.2, color="k")
fig.colorbar(pc, ax=ax, shrink=0.8).set_label("elevation of the middle of original layer 4 (m)")
ax.set(title="Original layer 4, sublayers averaged inside the pit", xlabel="x (m)", ylabel="y (m)")
ax.set_aspect("equal")
plt.show()

### The arrays for MODFLOW

`get_gridprops_mf6_disu()` returns the keyword arguments of `flopy.mf6.ModflowGwfdisu`, and `get_gridprops_mfusg_disu()` those of `flopy.mfusg.MfUsgDisU` (gridgen's key names); `write_gsf()` writes the grid specification file that MODFLOW-USG post-processors read. The dictionaries hold plain arrays:

In [ ]:
for name, gridprops in (("MF6", d5.get_gridprops_mf6_disu()), ("MODFLOW-USG", d5.get_gridprops_mfusg_disu())):
    print(name, {key: (f"{len(value)} entries" if hasattr(value, "__len__") else value) for key, value in gridprops.items()})
print("GSF written to", d5.write_gsf(workspace / "model.gsf"))

To build a model, pass them on:

```python
flopy.mf6.ModflowGwfdisu(gwf, **d5.get_gridprops_mf6_disu())
flopy.mf6.ModflowGwfnpf(gwf, k=k, xt3doptions=True, angle2=0.0, angle3=0.0)   # XT3D with vertical offsets
flopy.mfusg.MfUsgDisU(usg_model, **d5.get_gridprops_mfusg_disu())
```

**Notes:**

- **XT3D** (MODFLOW 6) is needed wherever connected cells are offset: dipping layers with full connectivity, and every change of plan grid, even with flat layers. Give NPF `angle2`/`angle3`.
- **MODFLOW-USG** has no XT3D; ghost-node corrections take its place. `disu_faults_and_gnc.ipynb` shows them, and how faults become barrier connections.
- **Merging small pieces** (`sliver_abs_tol`) is off by default: it gives fewer connections but moves face area between cells, so results are no longer exact.